# Stencil System — Mode B: OCR Decryption

This notebook demonstrates the **receiver-side OCR pipeline** (Mode B).

The sender encrypts a message and renders the obfuscated grid as a PNG.
The receiver captures that image (camera / CCTV) and recovers the plaintext using local TrOCR — no internet, no API key.

**Prerequisites**
```
# 1. Build the stencil_lib wheel
cd Implementation && inv build

# 2. Download TrOCR model once
python stencil_system_OCR/download_models.py
```

In [ ]:
import subprocess, sys, pathlib

dist = pathlib.Path('build/dist')
wheel = next(dist.glob('stencil_lib-*.whl'), None)
assert wheel, 'Run inv build first'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', str(wheel), '--force-reinstall'])

sys.path.insert(0, 'stencil_system_OCR')
import stencil_lib
print('stencil_lib ready')

## Step 1 — Sender: encrypt and render the stencil grid

In [ ]:
from stencil_lib import GridShape, StencilConfig, keygen, encrypt

# Caesar cipher, shift 3
class CaesarCfg:
    def __init__(self, shift): self.shift = shift

plaintext   = b'HELLO'
cipher_cfg  = CaesarCfg(shift=3)
grid_shape  = GridShape(n=2, shape=(10, 10))
stencil_cfg = StencilConfig(
    total_bytes=len(plaintext),
    num_partitions=len(plaintext),
    grid_shape=grid_shape,
    enable_cipher_permutation=False,
    enable_grid_permutation=False,
)

secret_key = keygen(stencil_cfg, cipher_cfg)
grid       = encrypt(plaintext, secret_key, stencil_cfg)
print('Grid encrypted:', len(grid.data), 'cells populated')

In [ ]:
from grid_renderer import render_grid
from IPython.display import display

img = render_grid(grid, grid_shape, output_path='/tmp/stencil_grid.png')
display(img)
print('Grid image saved to /tmp/stencil_grid.png')

## Step 2 — Receiver: OCR the grid image → reconstruct obfuscated Grid

In [ ]:
from ocr_pipeline import ocr_image_to_grid

reconstructed_grid = ocr_image_to_grid(
    '/tmp/stencil_grid.png',
    stencil_cfg,
    preprocess=False,
)
print(f'Reconstructed grid: {len(reconstructed_grid.data)} cells')

## Step 3 — Decrypt: stencil_lib handles everything internally

In [ ]:
recovered = stencil_lib.decrypt(reconstructed_grid, secret_key, stencil_cfg)
print('Original :', plaintext)
print('Recovered:', recovered)
print('Match    :', recovered == plaintext)

## One-call convenience

In [ ]:
from ocr_pipeline import full_ocr_decrypt

recovered = full_ocr_decrypt('/tmp/stencil_grid.png', secret_key, stencil_cfg, preprocess=False)
print('Recovered:', recovered)